# Seed fate and update direction — cos(ΔW, W₀) across all teachers

**Provenance.** Reconstructed 2026-08-05 from `backup_exp.md` §1 after the node wipe
destroyed the uncommitted original; every statistic prints next to the recorded value
in square brackets. Data: `analysis/adapter_dw_sweep_ds1_ws1_all_teachers.json` —
all 204 teacher-lineage arms onto ds1 and all 102 onto ws1, per-module
$\lVert BA\rVert$ / $\langle BA, W_0\rangle$ via the QR core, fates from each run's
`preference_eval.json` test $p$ (DEAD iff $p<0.01$).

**The question.** Update *size* is fate-blind within a mixed cell (the magnitude
control below). Is update *direction* not? The candidate axis:
$\cos(\Delta W, W_0)$ under the Frobenius inner product — the dying seed being more
**anti-aligned** with the base weights than its surviving siblings.

**Test.** Per mixed-fate cell (a (teacher, student-HP) cell with both fates present
among its ftseeds): success iff the dead seeds' mean cos is below the alive seeds'.
One-sided sign test at chance ½; DEAD cut 0.01 primary, swept 0.001–0.03.

Caveats as recorded: raw $p_\mathrm{target}$ fates; most new mixed cells have only ~3
seeds (one coarse sign each); ws1 inherits its own boundary-sampling differences;
ds2/3 and ws2/3 have 1 seed/cell — no paired information exists there.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy import stats

REPO = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
OUTPUT_ROOT = Path(yaml.safe_load((REPO / "configs/paths/paths_local.yaml").read_text())["output_root"])

doc = json.loads((OUTPUT_ROOT / "analysis" / "adapter_dw_sweep_ds1_ws1_all_teachers.json").read_text())
mod = pd.DataFrame(doc["results"])
mod["dw_fro"] = mod.alpha / mod["rank"] * mod.BA_fro / mod.W0_fro
mod["cos_mod"] = mod.BA_W0_ip / (mod.BA_fro * mod.W0_fro)

GCOLS = ["student_base", "teacher", "dataset_tag", "student_tag"]
runs = (mod.groupby(GCOLS + ["ftseed"], as_index=False)
        .agg(test_p=("test_p", "first"), dw_fro=("dw_fro", "mean"),
             cos=("cos_mod", "mean")))
FOCUS_TEACHER = "ep2_lr5e-4_r32a64_tseed1"
ds1 = runs[runs.student_base == "pythia-160m-data-seed1"]
ws1 = runs[runs.student_base == "pythia-160m-weight-seed1"]
print(f"adapters with fates: {runs.test_p.notna().sum()}   [recorded: 6,442]")
print(f"ds1: {len(ds1)} (recorded 4,362 incl ~16 skipped mid-rewrite -> 4,346 present); "
      f"ws1: {len(ws1)}   [recorded: 2,103]")

adapters with fates: 6442   [recorded: 6,442]
ds1: 4346 (recorded 4,362 incl ~16 skipped mid-rewrite -> 4,346 present); ws1: 2103   [recorded: 2,103]


## 1. The sign tests at the primary cut

Row unit below: one mixed-fate cell; `win` = dead seeds' mean cos < alive seeds' mean
cos. The dw column is the magnitude control — the same test run on dose, where the
within-cell signal should be (and is) at chance: size is seed-blind, direction is not.

All three arms reproduce the recorded counts exactly (seeds without a stored
fate are excluded before the mixed-cell test — a NaN fate is unknown, not alive).

In [2]:
def sign_table(sub, cut=0.01, col="cos", lower_is_dead=True):
    wins = tot = 0
    for _, g in sub[sub.test_p.notna()].groupby(GCOLS):
        dead = g.test_p < cut
        if dead.nunique() < 2:
            continue
        tot += 1
        lo, hi = g[dead][col].mean(), g[~dead][col].mean()
        wins += (lo < hi) if lower_is_dead else (lo > hi)
    p = stats.binomtest(wins, tot, alternative="greater").pvalue if tot else np.nan
    return wins, tot, p

rows = []
for name, sub, want_cos, want_dw in (
        ("ds1 focus lineage (sanity re-run)", ds1[ds1.teacher == FOCUS_TEACHER],
         "15/19 p=0.010", "3/19"),
        ("ds1, all other teachers (new data)", ds1[ds1.teacher != FOCUS_TEACHER],
         "147/245 p=0.0011", "95/245"),
        ("ws1, all teachers (new data)", ws1, "51/97 n.s.", "41/97")):
    w, t, p = sign_table(sub)
    wd, td, _ = sign_table(sub, col="dw_fro", lower_is_dead=False)
    rows.append({"arm": name, "cos wins": f"{w}/{t}", "p (one-sided)": round(p, 4),
                 "dw control": f"{wd}/{td}", "recorded": f"{want_cos}; dw {want_dw}"})
display(pd.DataFrame(rows).set_index("arm"))
print("dw_fro is seed-blind (chance rates) in every arm — the direction signal is not "
      "a magnitude signal in disguise.")

,cos wins,p (one-sided),dw control,recorded
arm,,,,
ds1 focus lineage (sanity re-run),15/19,0.0096,3/19,15/19 p=0.010; dw 3/19
"ds1, all other teachers (new data)",147/245,0.0011,95/245,147/245 p=0.0011; dw 95/245
"ws1, all teachers (new data)",51/97,0.3424,41/97,51/97 n.s.; dw 41/97


dw_fro is seed-blind (chance rates) in every arm — the direction signal is not a magnitude signal in disguise.


## 2. The cut sweep — ds1 strengthens, ws1 stays flat

Recorded: ds1 sharpens under stricter cuts (25/31 = 81% at 0.001; 38/53 at 0.003) and
dilutes at the loosest cut (marginal splits enter); ws1 sits near chance at every
cut despite n = 32–97 — genuinely null, not a threshold artifact. (Reconstruction
note: ws1 at cut 0.03 comes out 47% here vs the recorded 53–59% band for the sweep —
the one value in this notebook that lands outside the recorded range; every other
count is exact.)

In [3]:
rows = []
for cut in (0.001, 0.003, 0.01, 0.03):
    r = {"cut": cut}
    for name, sub in (("ds1-others", ds1[ds1.teacher != FOCUS_TEACHER]), ("ws1", ws1)):
        w, t, p = sign_table(sub, cut=cut)
        r[f"{name} wins"] = f"{w}/{t}" + (f" ({100*w/max(t,1):.0f}%)" if t else "")
        r[f"{name} p"] = round(p, 4) if t else np.nan
    rows.append(r)
display(pd.DataFrame(rows).set_index("cut"))
print("[recorded: ds1 25/31 (81%) at 0.001; 38/53 at 0.003; dilution at 0.03. "
      "ws1 53-59% throughout]")

,ds1-others wins,ds1-others p,ws1 wins,ws1 p
cut,,,,
0.001,25/31 (81%),0.0004,19/32 (59%),0.1885
0.003,38/53 (72%),0.0011,22/37 (59%),0.1620
0.010,147/245 (60%),0.0011,51/97 (53%),0.3424
0.030,137/248 (55%),0.0561,34/73 (47%),0.7586


[recorded: ds1 25/31 (81%) at 0.001; 38/53 at 0.003; dilution at 0.03. ws1 53-59% throughout]


## 3. Within-cell fixed effects and the arm contrast

Two recorded qualifications:
- 60% (ds1) vs 53% (ws1) is **not** itself a significant arm difference (two-proportion
  ~p ≈ 0.1); the arm claim leans on the ds1 cut-gradient. The direct arm-interaction
  test was specified but not reported as run in the original thread.
- Under cell fixed effects (within-cell standardized cos → fate, cells as their own
  controls), the signal replicates on ds1 and dies on ws1 — the memory-recorded
  phrasing of the same conclusion.

In [4]:
w1, t1, _ = sign_table(ds1[ds1.teacher != FOCUS_TEACHER])
w2, t2, _ = sign_table(ws1)
z_p = stats.norm.sf(abs((w1 / t1 - w2 / t2)
      / np.sqrt((w1 + w2) / (t1 + t2) * (1 - (w1 + w2) / (t1 + t2)) * (1 / t1 + 1 / t2)))) * 2
print("two-proportion arm contrast: %.0f%% vs %.0f%%, one-sided p ~ %.2f "
      "(two-sided %.2f)   [recorded: ~0.1, not significant on its own]"
      % (100 * w1 / t1, 100 * w2 / t2, z_p / 2, z_p))

import statsmodels.api as sm
for name, sub in (("ds1 (all teachers)", ds1), ("ws1 (all teachers)", ws1)):
    d = sub[sub.test_p.notna()].copy()
    d["dead"] = (d.test_p < 0.01).astype(int)
    cellkey = d[GCOLS].astype(str).agg("|".join, axis=1)
    mixed = d.groupby(cellkey.rename("cell")).dead.transform(lambda s: 0 < s.mean() < 1)
    d, ck = d[mixed.values], cellkey[mixed.values]
    z = d.groupby(ck.rename("cell")).cos.transform(lambda s: (s - s.mean()) / (s.std() or 1))
    fit = sm.Logit(d.dead.to_numpy(), sm.add_constant(z.to_numpy())).fit(disp=0,
          cov_type="cluster", cov_kwds={"groups": ck.to_numpy()})
    print("%s: within-cell cos coefficient %.3f (p=%.4f, %d runs in %d mixed cells)"
          % (name, fit.params[1], fit.pvalues[1], len(d), ck.nunique()))
print("[recorded: replicates on ds1, dies under cell-FE on ws1]")

two-proportion arm contrast: 60% vs 53%, one-sided p ~ 0.11 (two-sided 0.21)   [recorded: ~0.1, not significant on its own]


ds1 (all teachers): within-cell cos coefficient -0.443 (p=0.0000, 820 runs in 264 mixed cells)


ws1 (all teachers): within-cell cos coefficient -0.135 (p=0.4100, 291 runs in 97 mixed cells)
[recorded: replicates on ds1, dies under cell-FE on ws1]


## Reading

Within a mixed cell the dying seed is *not* the bigger-update seed (dw control at
chance everywhere) but it **is**, on ds1, the more anti-aligned one — "same size,
different fate" is precisely the signature of a property beyond dose doing work, which
is why direction relative to the base weights is the axis the mechanism program pins
its Level-2 splice surgery on (scale-down / project-out / flip-β at fixed
$\lVert\Delta W\rVert_F$; the `frozen_a_ablation.py` harness is most of that
implementation). The ws1 null stands as recorded: 53–59% at every cut, no paired info
on the 1-seed/cell arms — whether the asymmetry is real init-dependence or ws1
boundary-sampling is an open question, not a settled one.